# Semi-Markov SFSA: sampling-budget diagnostic

Reuse the existing full-training checkpoints; **no models are trained here**. Compare 500, 2,000 and 10,000 trajectories, with five sampling seeds, for Glazier-GRU, Glazier-RNN, Glazier-Transformer and the true-generator sampler. DeepHit and SurvTransformer remain fixed direct-prediction comparators.

Both Weibull and ReLU are included, at baseline and landmarks 10, 15 and 20. Every comparison uses the same test patients and reference curves. References are deterministic finite-state calculations, replacing the earlier 2,000-trajectory estimates.

**Start:** download `Glazier_SemiMarkov_sampling_budget.zip`. Locally, extract it and open this notebook from the extracted folder using your GPU environment. In Colab, select a GPU runtime, open this notebook, and run the cells in order; the setup cell asks for the ZIP if needed. The ZIP includes the previously trained local checkpoints. Load only checkpoints you trust.

The default `MODE="full"` executes the requested diagnostic and can take substantial time. Start with `"smoke"` to check setup. That checks execution only. Results are saved after each replicate and can resume with the same settings. Colab local storage is temporary; use the optional Drive cell for persistent results.

In [ ]:
# Must run before importing numerical libraries. On a local kernel already using
# another OpenMP runtime, restart the kernel before running this notebook.
import os, sys, datetime
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
if not hasattr(datetime, "UTC"):
    datetime.UTC = datetime.timezone.utc
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "lifelines==0.30.3", "matplotlib", "pandas"])


In [ ]:
from pathlib import Path
import zipfile

# Change this if the extracted bundle is stored elsewhere.
BUNDLE = Path.cwd()
candidates = [BUNDLE, BUNDLE / "Glazier_SemiMarkov_sampling_budget",
              BUNDLE / "outputs" / "Glazier_SemiMarkov_sampling_budget"]
found = next((p for p in candidates if (p / "sampling_budget.py").exists()), None)
if found is None and IN_COLAB:
    from google.colab import files
    print("Upload Glazier_SemiMarkov_sampling_budget.zip")
    uploaded = files.upload()
    archive = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
    if archive is None: raise FileNotFoundError("Please upload the bundle ZIP")
    destination = Path("/content/sampling_budget_bundle").resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            if not (destination / entry.filename).resolve().is_relative_to(destination):
                raise ValueError("Unsafe archive path")
        z.extractall(destination)
    found = destination / "Glazier_SemiMarkov_sampling_budget"
if found is None:
    raise FileNotFoundError("Extract the bundle ZIP, then set BUNDLE to its folder.")
BUNDLE = found.resolve()
# Optional workspace-only support packages for the original local gpu_env.
for parent in [BUNDLE, *BUNDLE.parents]:
    support = parent / "work" / "gpu_support"
    if support.exists():
        sys.path.insert(0, str(support)); break
sys.path.insert(0, str(BUNDLE / "source"))
sys.path.insert(0, str(BUNDLE))
import torch, numpy as np, pandas as pd
from sampling_budget import run, plot_results
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
print("Bundle:", BUNDLE)
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU; full mode may be slow")


## Optional persistent Colab output
Set `USE_DRIVE=True` to save results to Google Drive before starting the long run. The cohort and checkpoint bundle stays unchanged. Changing experiment settings requires a different output directory.

In [ ]:
USE_DRIVE = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/Glazier_sampling_diagnostics")
else:
    OUTPUT_PARENT = BUNDLE / "results"


## Experiment settings
`full`: 300 held-out patients, three budgets and five seeds. Landmark subsets retain only those alive and uncensored at that landmark. `smoke`: three test patients, two small budgets and two seeds, with baseline and landmark 10. The five full-mode seeds change **prediction sampling only**, not training.

Budgets use distinct deterministic random streams; samples are not nested across budgets. Curves are saved per replicate, and existing matching replicates are reused. The configuration includes source and checkpoint hashes to prevent accidentally mixing experiments.

In [ ]:
MODE = "full"  # change to "smoke" for a quick execution check
SCENARIOS = ["Weibull", "ReLU"]
MODELS = ["Glazier-GRU", "Glazier-RNN", "Glazier-Transformer"]
if MODE == "full":
    BUDGETS = [500, 2000, 10000]
    SAMPLING_SEEDS = [41001, 52001, 63001, 74001, 85001]
    N_EVAL = 300
    LANDMARKS = [0, 10, 15, 20]  # 0 means baseline
elif MODE == "smoke":
    BUDGETS = [32, 64]
    SAMPLING_SEEDS = [41001, 52001]
    N_EVAL = 3
    LANDMARKS = [0, 10]
else:
    raise ValueError("MODE must be full or smoke")
OUTPUT = OUTPUT_PARENT / (MODE + "_sampling_budget_v1")
print("Saving to:", OUTPUT)
print("Budgets:", BUDGETS, "Seeds:", SAMPLING_SEEDS)


## Run the diagnostic
The oracle sampler uses the true generator at the same trajectory budgets; it is a finite-sampling control, **not an information lower bound**. The deterministic reference corresponds to the preserved observed-generator, event-first KM functional. It does not resolve the paper's latent-event censoring interpretation. References assume the bundled default generator; custom generator changes require matching reference changes.

In [ ]:
replicates, summary, comparisons = run(
    BUNDLE, OUTPUT, scenarios=SCENARIOS, budgets=BUDGETS,
    seeds=SAMPLING_SEEDS, n_eval=N_EVAL, landmarks=LANDMARKS,
    device=DEVICE, models=MODELS,
)
display(summary)


## Plot error versus sampling variability
Left: mean curve MAE across sampling seeds, with **±1 seed standard deviation**, not confidence intervals. Dashed lines show the fixed direct-model errors. Right: square root of the average pointwise variance across sampling seeds. Falling right-hand curves show reduced sampling noise even if overall error eventually plateaus.

In [ ]:
summary = plot_results(OUTPUT)
display(pd.read_csv(OUTPUT / "direct_comparisons.csv"))


## Separate finite-replicate error components
For the saved replicate curves, mean squared error equals error of their mean curve plus their finite-replicate spread. This identity is checked numerically. `ensemble_MSE` is **not pure model bias**: the mean of five curves still contains sampling error and finite-sample KM effects. `sampling_RMS` estimates variability conditional on these checkpoints and patients.

MAE has no corresponding additive decomposition; do not subtract oracle MAE from Glazier MAE and call the difference learning error. Repeated sampling seeds do not measure training-seed or patient-selection uncertainty.

In [ ]:
display(summary[["scenario", "landmark", "model", "budget", "mean_draw_MSE",
                 "ensemble_MSE", "finite_replicate_sampling_component", "sampling_RMS", "ensemble_MAE"]])
assert np.allclose(summary.mean_draw_MSE,
                   summary.ensemble_MSE + summary.finite_replicate_sampling_component)


## Interpret the patterns
- Glazier approaches the direct-model error as trajectories increase: finite sampling contributed to the original gap.
- Glazier and the oracle improve similarly: the original error was substantially sampling-related, though proximity alone does not prove transition-rule recovery.
- Oracle error keeps falling while Glazier plateaus: investigate learned-distribution error, calibration, or objective alignment.
- Both flatten: check whether the remaining range and number of sampling seeds can resolve the difference; inspect saved curves and numerical conventions.
- Landmark direct comparators still use survival ratios and baseline covariates; Glazier receives the observed prefix. This experiment does not make their input information equivalent.

The diagnostic measures the checkpoints already trained, not an architecture's ultimate capacity. Exact references remove reference Monte Carlo noise; they do not remove uncertainty in training or the test population.

## Save and download results
The results include individual curves, deterministic references, patient IDs, configuration/checkpoint hashes, replicate metrics, summary comparisons, and PNG plots. The archive is created outside the results folder to avoid including itself.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT.parent / (OUTPUT.name + "_results")), "zip", root_dir=OUTPUT)
print("Results archive:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
